     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.2/102.2 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.0/105.0 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 62.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 5.8 MB/s eta 0:00:00


📂 Loading full dataset from: Lenskart India.csv...
✅ Loaded and sanitized 150,000 rows successfully.
🔌 Establishing secure connection to Snowflake...
❌ Pipeline Execution Error: 002043 (02000): SQL compilation error:
Object does not exist, or operation cannot be performed.
🔒 Snowflake connection closed safely.


In [2]:
# 1. Install production dependencies
!pip install snowflake-connector-python[pandas] pandas

import time
import snowflake.connector
import pandas as pd
from google.colab import files
from snowflake.connector.pandas_tools import write_pandas

# --- CONFIGURATION ---
SNOWFLAKE_USER = "JOHNATHANAJIT007"
SNOWFLAKE_PASSWORD = "9392340168MAenglish"  # <--- Updated with your correct password
SNOWFLAKE_ACCOUNT = "fmesfxh-hu66093"
SNOWFLAKE_WAREHOUSE = "COMPUTE_WH"
SNOWFLAKE_DATABASE = "REALTIME_ECOMMERCE_DB"
SNOWFLAKE_SCHEMA = "TELEMETRY"
TARGET_TABLE = "RAW_LENSKART_TELEMETRY"

def sanitize_columns(df):
    """Sanitize DataFrame columns for flawless Snowflake DDL mapping."""
    df.columns = (
        df.columns.str.strip()
        .str.upper()
        .str.replace(' ', '_')
        .str.replace('-', '_')
        .str.replace('.', '_')
        .str.replace('(', '')
        .str.replace(')', '')
    )
    return df

def run_telemetry_pipeline():
    pipeline_start = time.time()

    # Step 1: Upload and Load Dataset
    print("📂 Please upload your 'Lenskart India.csv' file when prompted...")
    uploaded = files.upload()

    # Automatically grab the filename you just uploaded
    filename = list(uploaded.keys())[0]
    print(f"📂 Loading dataset from: {filename}...")

    df = pd.read_csv(filename)
    df = sanitize_columns(df)
    print(f"✅ Loaded and sanitized {len(df):,} rows successfully.")

    # Step 2: Establish Secure Snowflake Connection
    print("🔌 Establishing secure connection to Snowflake...")
    conn = snowflake.connector.connect(
        user=SNOWFLAKE_USER,
        password=SNOWFLAKE_PASSWORD,
        account=SNOWFLAKE_ACCOUNT,
        warehouse=SNOWFLAKE_WAREHOUSE,
        database=SNOWFLAKE_DATABASE,
        schema=SNOWFLAKE_SCHEMA
    )

    try:
        # Explicitly set session context
        cursor = conn.cursor()
        cursor.execute(f"USE WAREHOUSE {SNOWFLAKE_WAREHOUSE}")
        cursor.execute(f"USE DATABASE {SNOWFLAKE_DATABASE}")
        cursor.execute(f"USE SCHEMA {SNOWFLAKE_SCHEMA}")
        cursor.close()

        # Step 3: High-Performance Bulk Upload
        print(f"🚀 Streaming full dataset into Snowflake table `{TARGET_TABLE}`...")

        success, nchunks, nrows, output = write_pandas(
            conn=conn,
            df=df,
            table_name=TARGET_TABLE,
            auto_create_table=True,
            overwrite=True
        )

        if success:
            elapsed_time = time.time() - pipeline_start
            print("\n" + "="*50)
            print("✨ PRO-GRADE PIPELINE EXECUTION SUCCESSFUL ✨")
            print(f"📊 Total Rows Ingested : {nrows:,}")
            print(f"📦 Chunks Transferred  : {nchunks}")
            print(f"⏱️ Total Execution Time: {elapsed_time:.2f} seconds")
            print("="*50)
        else:
            print("❌ Data transfer encountered an issue during write_pandas execution.")

    except Exception as e:
        print(f"❌ Pipeline Execution Error: {e}")

    finally:
        conn.close()
        print("🔒 Snowflake connection closed safely.")

if __name__ == "__main__":
    run_telemetry_pipeline()

📂 Please upload your 'Lenskart India.csv' file when prompted...


Saving Lenskart India.csv to Lenskart India.csv
📂 Loading dataset from: Lenskart India.csv...
✅ Loaded and sanitized 150,000 rows successfully.
🔌 Establishing secure connection to Snowflake...
🚀 Streaming full dataset into Snowflake table `RAW_LENSKART_TELEMETRY`...

✨ PRO-GRADE PIPELINE EXECUTION SUCCESSFUL ✨
📊 Total Rows Ingested : 150,000
📦 Chunks Transferred  : 1
⏱️ Total Execution Time: 140.28 seconds
🔒 Snowflake connection closed safely.


In [3]:
import snowflake.connector

# --- CONFIGURATION ---
SNOWFLAKE_USER = "JOHNATHANAJIT007"
SNOWFLAKE_PASSWORD = "9392340168MAenglish"
SNOWFLAKE_ACCOUNT = "fmesfxh-hu66093"
SNOWFLAKE_WAREHOUSE = "COMPUTE_WH"
SNOWFLAKE_DATABASE = "REALTIME_ECOMMERCE_DB"
SNOWFLAKE_SCHEMA = "TELEMETRY"
TARGET_TABLE = "RAW_LENSKART_TELEMETRY"

print("🔌 Connecting to Snowflake for data verification...")
conn = snowflake.connector.connect(
    user=SNOWFLAKE_USER,
    password=SNOWFLAKE_PASSWORD,
    account=SNOWFLAKE_ACCOUNT,
    warehouse=SNOWFLAKE_WAREHOUSE,
    database=SNOWFLAKE_DATABASE,
    schema=SNOWFLAKE_SCHEMA
)

cursor = conn.cursor()

try:
    # 1. Verify Total Row Count
    print(f"\n📊 Checking row count for table `{TARGET_TABLE}`...")
    cursor.execute(f"SELECT COUNT(*) FROM {TARGET_TABLE}")
    row_count = cursor.fetchone()[0]
    print(f"✅ Total Verified Rows in Snowflake: {row_count:,}")

    # 2. Preview Schema / Columns
    print(f"\n📋 Previewing table columns and sample record:")
    cursor.execute(f"SELECT * FROM {TARGET_TABLE} LIMIT 1")
    columns = [col[0] for col in cursor.description]
    sample_row = cursor.fetchone()

    print(f"Total Columns: {len(columns)}")
    print(f"Columns: {columns[:5]} ... (and {len(columns)-5} more)")

except Exception as e:
    print(f"❌ Verification Error: {e}")

finally:
    cursor.close()
    conn.close()
    print("\n🔒 Verification complete. Connection closed safely.")

🔌 Connecting to Snowflake for data verification...

📊 Checking row count for table `RAW_LENSKART_TELEMETRY`...
✅ Total Verified Rows in Snowflake: 150,000

📋 Previewing table columns and sample record:
Total Columns: 14
Columns: ['ORDER_ID', 'ORDER_DATE', 'CITY', 'STATE', 'SALES_CHANNEL'] ... (and 9 more)

🔒 Verification complete. Connection closed safely.
